- DataFrame         🟢
- Explicit schema       🟢
- Filtering        🟢 
- NULL handling         🟢
- Duplicate removal     🟢
- Date transformations      🟢
- Aggregation       🟢
- GroupBy + aggregation         🟢
- Having condition      🟢
- Highest/second-highest salary     🟢
- Monthly aggregation
- Window        🟢
- Latest transaction per customer
- Top 3 per group
- row_number
- rank
- dense_rank
- lag
- lead
- Running total
- Joins
- Inner join
- Left join
- Anti join
- Broadcast join
- Join with duplicate columns

**Complex**

- Explode
- Flatten JSON
- Pivot/unpivot
- ETL
- Incremental load
- Deduplication + latest record
- SCD Type 1
- SCD Type 2
- Delta MERGE

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, TimestampType, BooleanType, ArrayType, MapType

# Employees Table: For Salary/window/groupBy/etc.
employee_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("dept_id", IntegerType(), True),
    StructField("salary", DoubleType(), True),
    StructField("hire_date", StringType(), True),
    StructField("status", StringType(), True)
])

employee_data = [
    (1, "Alice", 10, 7000.0, "2022-01-05", "Active"),
    (2, "Bob", 20, 9000.0, "2022-03-10", "Active"),
    (3, "Charlie", 10, 7000.0, "2022-02-15", "Active"),
    (4, None, 20, None, "2022-04-07", "Left"),
    (5, "Eve", 30, 12000.0, "2021-12-22", "Active"),
    (6, "Frank", 10, 9000.0, "2022-02-22", "Active"),
    (7, "Grace", 30, 12000.0, "2021-12-22", "Active"), # duplicate salary/date
    (8, "Heidi", 20, 9500.0, "2023-06-01", "Active"),
    (3, "Charlie", 10, 7000.0, "2022-02-15", "Active"), # duplicate emp_id
]

emp_df = spark.createDataFrame(employee_data, schema=employee_schema)

# Departments Table: For join practice
dept_schema = StructType([
    StructField("dept_id", IntegerType(), False),
    StructField("dept_name", StringType(), True),
])

dept_data = [
    (10, "HR"),
    (20, "Engineering"),
    (30, "Sales"),
    (40, "Marketing")
]

dept_df = spark.createDataFrame(dept_data, schema=dept_schema)

# Transactions Table: For incremental/window/ETL/Top N/groupBy/Having
transaction_schema = StructType([
    StructField("txn_id", IntegerType(), False),
    StructField("emp_id", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("txn_type", StringType(), True),
    StructField("txn_date", StringType(), True)
])

transaction_data = [
    (101, 1, 500.0, "expense", "2023-01-15 10:15:00"),
    (102, 2, 1000.0, "expense", "2023-01-20 11:22:00"),
    (103, 1, 300.0, "expense", "2023-02-18 09:05:00"),
    (104, 3, 1500.0, "income", "2023-03-05 12:00:00"),
    (105, 4, None, "expense", "2023-03-10 14:03:00"),
    (106, 5, 700.0, "expense", None),
    (107, 2, 850.0, "income", "2023-03-22 17:30:00"),
    (108, 1, 950.0, "income", "2023-04-01 10:00:00"),
    (109, 5, 1200.0, "expense", "2023-04-02 11:45:00"),
    (110, 1, 1300.0, "income", "2023-04-03 09:50:00")
]

tran_df = spark.createDataFrame(transaction_data, schema=transaction_schema)

# Complex Table: For explode/flatten/pivot/unpivot/SCD/json
complex_schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("event_date", StringType(), True),
    StructField("tags", ArrayType(StringType()), True),
    StructField("meta", MapType(StringType(), StringType()), True),
    StructField("profile", StringType(), True) # JSON string
])

complex_data = [
    (1, "2023-06-10", ["new","vip"], {"device":"mobile","city":"NY"}, '{"email":"alice@acme.com","age":30,"active":true}'),
    (2, "2023-06-11", None, {"device":"web"}, '{"email":"bob@acme.com","age":28,"active":false}'),
    (3, "2023-06-12", ["regular"], None, '{"email":"eve@acme.com","age":25,"active":true}'),
    (4, "2023-06-13", [], {"device":"mobile"}, '{"email":"charlie@acme.com","age":34,"active":false}')
]

complex_df = spark.createDataFrame(complex_data, schema=complex_schema)

# Customers Table: For SCD/Delta practice
customer_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("valid_from", StringType(), True),
    StructField("valid_to", StringType(), True),
    StructField("current_flag", BooleanType(), True)
])

customer_data = [
    (1, "MegaCorp", "NY", "2022-01-01", "2024-01-01", False),
    (1, "MegaCorp", "LA", "2024-01-01", None, True),
    (2, "StartupX", "SF", "2023-01-01", None, True),
    (3, "AlphaTech", "TX", "2024-01-01", None, True)
]

cust_df = spark.createDataFrame(customer_data, schema=customer_schema)

# Show DataFrames to verify
display(emp_df)
display(dept_df)
display(tran_df)
display(complex_df)
display(cust_df)

In [0]:
"""
DataFrame ---
Explicit schema  ---
Filtering  ---
NULL handling ---
Duplicate removal ---
Date transformations  ---

"""


from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

emp_df = emp_df.fillna({"name" :"unknown","salary" : 0})
window_spec = Window.partitionBy(col("emp_id")).orderBy(col("hire_date"))
emp_df = emp_df.withColumn("rnk",rank().over(window_spec))\
    .filter(col("rnk")==1)\
    .drop("rnk")
display(emp_df)

In [0]:
'''
Aggregation --
GroupBy + aggregation ---
Having condition ---
Highest/second-highest salary ---
'''
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = emp_df.join(dept_df, "dept_id", "inner")\
    .filter(col("status")=="Active")\
    .agg(avg(col("Salary")))
#df.show()

df1= emp_df.alias("a").join(dept_df.alias("b"),"dept_id", "inner")\
    .groupBy("b.dept_name")\
    .agg(avg("salary").alias("Avegrage_salary"))\
    .filter(col("Avegrage_salary")>=7000)
#df1.show()

#emp_df.show()
#dept_df.show()

window_spec = Window.orderBy(desc("salary"))
max_sal = emp_df.withColumn("rnk",dense_rank().over(window_spec))\
    .filter(col("rnk")==2)\
    .select("emp_id","name","dept_id","salary")
max_sal.show()
